In [ ]:
%pip install pandas statsmodels numpy

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# Logistic Regression

Логистическая регрессия является модификацией линейной регрессии
Линейная регрессия строит уравнение вида LinReg_y = f(X) +  ε
Логистическая регрессия применяет логистическую функцию к LinReg_y
$$Sigmoid\_function\_result = ln(\frac{1}{1 + e^{-LinReg\_y}})$$

Если Sigmoid_function_result >= 0.5, то результат ```1``` \
Иначе результат ```0```

### At this assignment you can use any tool, language and/or library. But it is strongly advised to try in-database analysis with MADlib, in that case you should have PostgreSQL database and should ignore all remarks about library outdated version in VMs - just follow up to date documentation. 

## 4.4        (Logistic regression)  If one predicts the probability of success of all the data points in a training set using a logistic regression model and adds up all of these probabilities, this number will equal the number of successes in the training set.  In other words, “the probability mass equals the counts.”  In this assignment, you will analyze this fact by using the same survey.csv dataset that was used is in Lab Exercise 7: Logistic Regression. Do this by creating a Postgresql query that uses MADlib functions.

### Create a Postgresql query using MADlib functions

###   As done in Lab Exercise 6: Linear Regression, create a column in the survey table that consists of the integer 1 for all of the rows of data. 

In [ ]:
raw = pd.read_csv("resources/5-lab/survey.csv")
raw['const'] = 1
raw

### Since the price column’s values are categorical between \\$10, \\$20, and \\$30, create the following additional columns in the survey table:

### - price20 - will have the value 1 if the price is $20, 0 otherwise

### - price30 - will have the value 1 if the price is $30, 0 otherwise

В столбце ```Price``` всего 3 варианта цены (10, 20, 30), потому с помощью one-hot encoding переведем из чисел в ортогональные вектора
- \\$10 ~ (0, 0)
- \\$20 ~ (0, 1)
- \\$30 ~ (1, 0)

In [ ]:
raw['price20'] = (raw['Price'] == 20).astype(int)
raw['price30'] = (raw['Price'] == 30).astype(int)
raw = raw.drop(columns=['Price'])
raw

### In order to create a logistic model on the data, the dependent variable (mydepv) needs to be of type Boolean while all of the independent variables need to be in the form of an array. 

In [ ]:
raw['MYDEPV'] = raw['MYDEPV'].astype(bool)
df = raw
del(raw)
df

In [ ]:
X = df[['const', 'Income', 'Age', 'price20', 'price30']]
Y = df['MYDEPV']

## a.  Use the MADlib function logregr to create a logistic regression model and display only the coefficients of the independent variables.  Note: You can refer to the documentation for logregr at:

In [ ]:
logreg = sm.Logit(Y, X)
result = logreg.fit()

print(result.params)

## b.      For every unit increase in income while all other independent variables remain constant, by what percentage does the odds-ratio increase/decrease?

Как считается шанс
$$ odd = \frac{P}{1 - P}$$



Посчитаем $$\Large (e^{b_{income}} - 1)* 100 \% $$

При повышении дохода на единицу, шансы повысятся на 13.7%

In [ ]:
print(f"{(np.e ** result.params['Income'] - 1) * 100:.1f}%")

### c.       If the price of a row of data were to increase from \\$10 to $30 while all other independent variables remained constant, by what percentage would the odds-ratio increase/decrease?

Поскольку 10 долларов это вектор (0 , 0)\
То для оценки ситуации перехода из 10 в 30 долларов достаточно лишь price30 обратить в единицу

То есть оценить, на сколько изменится отношение шансов при повышении price30 на единицу

In [ ]:
print(f"{(np.e ** result.params['price30'] - 1) * 100:.1f}%")

## Use our logistic regression model to make predictions on probability of success for each of the rows of data in the survey table. Add a column named odds_ratio to the survey table that calculates the odds-ratio of each row of data. Add another column to the survey table named prediction that uses the odds_ratio column data to solve for the probability of success for a row of data.

$$\Large odds\_ratio = e^{LinReg\_y} $$
$$\Large P = \frac{odds\_ratio}{1 + odds\_ratio}$$


In [ ]:
result.params

In [ ]:
LinReg_y = df[['const', 'Income', 'Age', 'price20', 'price30']] @ result.params
LinReg_y

In [ ]:
df['odds_ratio'] = np.exp(LinReg_y)
df['prediction'] = df['odds_ratio'] / (1 + df['odds_ratio'])

# Проверка, что я правильно посчитал
pred = result.predict(X)
if abs(df['prediction'] - pred).max().round(15) != 0:
    print("ERROR")

df

### d.      Test the rule that the probability mass equals the counts.  Use the survey table and take the sums of the mydepv column and the prediction column.  Are the values equal? Excluding rounding errors, does probability mass equal count?
Проверим уравнение градиента функции правдоподобия
$$\sum_{i=1}^{n} y_i = \sum_{i=1}^{n} {p}_i$$


In [ ]:
actual = df['MYDEPV'].sum()
predicted = df['prediction'].sum()

print(f"Проверка уравнения: {actual == predicted}")

### e.       Using the logistic model we have created, what is the likelihood of a person who is 25 years old with an income of \\$58,000 accepting an offer with a price of $20?


In [ ]:
print(f"Вероятность, что примет предложение: {result.predict([[1, 58, 25, 1, 0]])[0].round(2)}")

In [ ]:
## Нужно ли делать для зачета Оксфордскую часть??